# The A/B Test You Checked Too Often

**Topic:** Multiple testing &nbsp;·&nbsp; **Difficulty:** Medium

Your A/B dashboard refreshes every hour and shows a p-value. The temptation is to
stop the moment it dips below 0.05. This is the single most common way to ship a
change that does nothing.

Simulate it. Run experiments where the two variants are *identical*, peek
repeatedly, and count how often you would have declared victory. The nominal 5%
false-positive rate is not what you get.

## The setup (constants in the next cell, with the data drawn in your solution)

- `n_sims = 4000` simulated experiments, all with **zero true effect**.
- Each experiment collects 20 batches of 200 observations per arm (4,000 per arm).
- Use `np.random.default_rng(777)`.

## Your task

With **NumPy only**:

1. For each simulation, generate both arms from the *same* standard normal
   distribution: draw all 4,000 observations of arm A as one
   `rng.normal(0, 1, (n_sims, 4000))` block, then all 4,000 of arm B the same
   way.
2. After each batch, compute the two-sample z statistic on the data collected so
   far: `(mean_a - mean_b) / sqrt(var_a/n + var_b/n)` using `np.var(..., ddof=1)`.
3. `stopped_early`: the fraction of simulations where `|z| > 1.96` at **any**
   of the 20 checkpoints.
4. `final_only`: the fraction where `|z| > 1.96` at the **final** checkpoint, counting a simulation even if it also crossed at an earlier peek.
5. Set `answers` to a dict with `"stopped_early"`, `"final_only"`, and
   `"inflation"` = `stopped_early / final_only`.

## Evaluation

Compared to the reference to a relative tolerance of 1e-6 (the seed makes this
exact). The evaluation cell also checks the peeking rate is well above 0.05.

In [ ]:
import numpy as np

n_sims = 4000
n_checks = 20
batch = 200
print(f"{n_sims} simulations, {n_checks} peeks, {batch} obs per arm per batch")
print("true effect size: 0.0. Every 'significant' result here is a false positive.")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Simulate the experiments described above, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("stopped_early", [], 0.25425, -0.042571115266586405, 1e-06),
    ("final_only", [], 0.05075, -0.008497479251835832, 1e-06),
    ("inflation", [], 5.009852216748768, -0.8388397096864316, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "stopped_early" in answers:
    _bad = answers["stopped_early"] > 0.12
    print(f"  {'peeking inflates alpha':<24} {'ok' if _bad else 'wrong'}  "
          f"(rate = {answers['stopped_early']:.3f} vs nominal 0.05)")
    _ok = _ok and _bad

print("=" * 56)
if _ok:
    print("PASSED. Nice work!")
else:
    print("NOT PASSED yet. Revisit your solution and try again.")

## Solution

Generate the whole `(n_sims, n_checks * batch)` block per arm once, then use
`cumsum` to get running means and running sums of squares, which gives every
checkpoint without re-summing. Each individual test is valid at 5%. It is the
*maximum* over 20 correlated tests that blows past it, landing around 25%.

```python
import numpy as np

rng = np.random.default_rng(777)
total = n_checks * batch

a = rng.normal(0, 1, (n_sims, total))
b = rng.normal(0, 1, (n_sims, total))

ns = np.arange(batch, total + 1, batch)

ca, cb = np.cumsum(a, axis=1), np.cumsum(b, axis=1)
ca2, cb2 = np.cumsum(a ** 2, axis=1), np.cumsum(b ** 2, axis=1)

cols = ns - 1
sa, sb = ca[:, cols], cb[:, cols]
sa2, sb2 = ca2[:, cols], cb2[:, cols]

ma, mb = sa / ns, sb / ns
va = (sa2 - ns * ma ** 2) / (ns - 1)
vb = (sb2 - ns * mb ** 2) / (ns - 1)

z = (ma - mb) / np.sqrt(va / ns + vb / ns)
hit = np.abs(z) > 1.96

stopped_early = float(hit.any(axis=1).mean())
final_only = float(hit[:, -1].mean())

answers = {
    "stopped_early": stopped_early,
    "final_only": final_only,
    "inflation": stopped_early / final_only,
}
```